# Start here

PSBD-ViT adapts Prediction Shift Backdoor Detection, a backdoor detector published for ResNets, to Vision Transformers. The defender never sees the training data or the trigger. It holds a trained model and a small clean validation set, and it has to decide, one image at a time, whether that image carries a trigger. PSBD's answer is to run the model several times with a perturbation switched on, compare each perturbed prediction against the unperturbed one, and read a shifting prediction as a sign of a clean decision working properly. A decision that does not shift is read as resting on a shortcut, which is what a trigger gives a model.

This notebook is the index. It explains the vocabulary every other notebook uses, shows the two commands that produce a detection number, and states which file on disk holds which number, so a later notebook can read results rather than recompute them.

In [1]:
import os
import sys
from pathlib import Path

# Anchor at the repository root so every default path in the library resolves the
# same way it does from a script, whichever directory the notebook was opened from.
REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import logging

logging.getLogger("lightning.fabric.utilities.seed").setLevel(logging.WARNING)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import scripts.paper._style  # noqa: F401  the figure style every table in the paper uses
print('repository root:', REPO_ROOT)

repository root: /lustre/home/pstika/projects/PSBD-ViT


## Vocabulary

A **site** is a named tensor boundary inside a transformer block, reached through a forward hook, so no trained weight is touched and the model's own dropout stays off. A **perturbation** is what the hook does to the tensor at that site: dropout, token masking (zero whole tokens, rescale the rest), channel masking, additive Gaussian noise, or gain scaling of a LayerNorm output. A **placement** names a site together with a perturbation, for example `before_attention_norm_token_mask`. A **rate** is the perturbation's own strength parameter, swept over a ladder from 0.05 to 0.9.

A **sweep** runs one placement on one model over its rate ladder and stores every per-pass prediction. An **analysis** reads that cache and writes the detection metrics: AUROC, true-positive rate and false-positive rate under several threshold rules. The **basis** is the declared set of placements swept on every model, listed in `configs/psbd_basis.json`. A **band** restricts a placement to a range of transformer blocks, and a placement with no band acts in every block.

Two placements carry names of their own, because the paper's whole argument is a comparison between them. **PSBD-RD** reads dropout on the residual stream after both residual adds, the closest a ViT block comes to the original ResNet placement, which applied dropout after the residual add inside the basic block. **PSBD-TM** masks whole tokens at the attention input, before the LayerNorm that feeds self-attention. Section 5 of the paper reads PSBD-TM at a mean AUROC of 0.935 against 0.832 for PSBD-RD, and notebook 04 works through why.

The detection statistic is the prediction shift uncertainty, PSU: the drop in the probability the model assigned its own unperturbed prediction, averaged over the perturbed passes. A low PSU means the prediction barely moved under the perturbation, and the decision rule is one-sided, so a low PSU is read as poisoned. The rate a placement is read at is chosen by a rule stated in advance, never picked after seeing the detection numbers: the smallest rate whose clean-validation predictions shift 80 percent of the time. `defences.decision.select_rate_adaptively` implements this, and `ADAPTIVE_SHIFT_TARGET` in the same module is the constant 0.8.

In [2]:
from defences.decision import (
    ADAPTIVE_SHIFT_TARGET,
    HEADLINE_QUANTILE,
    PLACEMENT_MATCH_TARGET,
    PUBLISHED_PLACEMENT,
    RECOMMENDED_PLACEMENT,
)

print(f"deployable rate rule targets a shift ratio of {ADAPTIVE_SHIFT_TARGET}")
print(f"cross-placement comparisons are read at a matched shift ratio of {PLACEMENT_MATCH_TARGET}")
print(f"clean-validation threshold sits at the {HEADLINE_QUANTILE:.0%} quantile")
print(f"PSBD-TM placement id: {RECOMMENDED_PLACEMENT}")
print(f"PSBD-RD placement id: {PUBLISHED_PLACEMENT}")

deployable rate rule targets a shift ratio of 0.8
cross-placement comparisons are read at a matched shift ratio of 0.6
clean-validation threshold sits at the 25% quantile
PSBD-TM placement id: before_attention_norm_token_mask
PSBD-RD placement id: post_residual


## Running one sweep and one analysis

Every detection number in the paper traces back to these two commands, run from the repository root with the virtual environment active:

```bash
python -m cli.sweep \
    --checkpoint checkpoints/vit_cifar100_badnet_a2o_0_01 \
    --position before_attention_norm --operator token_mask
python -m cli.analyze --folder vit_cifar100_badnet_a2o_0_01
```

`cli.sweep` needs a GPU. It loads the checkpoint, attaches the placement, runs the perturbed passes at every rate on the ladder, and writes the raw per-pass probabilities under `results/<folder>/psbd/<placement>/`, with a `run_<placement>.json` file recording the rate ladder and the number of passes. `cli.analyze` runs on CPU. It reads that cache, applies every threshold rule, and writes `results/<folder>/psbd_metrics.json`, one entry per placement. Notebook 02 runs both commands' logic directly, on one model.

## Which command reads which file

| file | written by | read by |
|---|---|---|
| `checkpoints/<folder>/attack_result.pt`, `args.json` | `cli.train_backdoor`, `cli.train_benign` | every command below |
| `checkpoints/<folder>/metrics.json` | `cli.evaluate` | notebook 01, the coverage ledger |
| `results/<folder>/psbd/<placement>/*.pt` | `cli.sweep` | `cli.analyze` |
| `results/<folder>/psbd_metrics.json` | `cli.analyze` | `scripts/paper/*.py`, notebooks 02 to 06 |
| `results/coverage/coverage.json` | the coverage ledger script | `scripts/paper/*.py`, notebook 06 |
| `paper/tables/*.tex`, `*.macros.json` | `scripts/paper/tab_*.py`, `fig_*.py`, `mech_*.py` | `paper/headline.tex`, notebook 06 |

Every notebook past this one reads from `results/` rather than recomputing a sweep, except where the point of the notebook is to run the method itself on one model.

## Reading order

01 loads a dataset and applies every attack's trigger. 02 runs PSBD-TM and PSBD-RD end to end on one checkpoint. 03 reads the staircase tables that isolate the site from the perturbation. 04 asks why the ranking looks the way it does, by tracing a trigger through a block. 05 transfers the method to Swin and tests an attacker who knows the defence exists. 06 shows how the whole paper is regenerated from `results/` by one script, and states what is single-seed and what is still running.